# 02 · Modèle en flocon : dimensions, calendrier et contexte

**Projet 4 : Analyse des données de vente et retours** · Sprint 1 · US10

**Objectif** : préparer les tables de dimension et les rattacher aux ventes, pour pouvoir analyser les ventes par boutique, pays, produit, catégorie, promotion, canal et date.

| Étape | Ce qu'on fait |
|---|---|
| 1. Chargement | Lire les ventes consolidées (`01_etl`) et les tables de dimension brutes |
| 2. Réalignement des dates | Décaler de +12 ans les dates des dimensions Contoso (2007-2009 → 2019-2021) |
| 3. Dimension boutique | Joindre boutique et géographie : chaque boutique reçoit sa ville, sa région et son **pays** |
| 4. Dimension produit | Aplatir la hiérarchie produit → sous-catégorie → catégorie |
| 5. Dimensions promotion et canal | Garder les colonnes utiles, avec les dates réalignées |
| 6. Dimension date | Construire notre calendrier 2019-2021 (commun à tous les pays) |
| 7. Contexte par pays et par jour | Jours fériés et sévérité COVID-19 : **une seule table**, avec une colonne `pays` |
| 8. Jointure aux ventes | Rattacher à chaque vente ses attributs, puis le contexte de son pays ce jour-là |
| 9. Enregistrement | Écrire les tables dans `data/interim` |
| 10. Modèle complet | Situer les ventes parmi les cinq tables de faits du projet et leurs dimensions partagées |

**Lit dans** : `data/interim/ventes_consolidees.parquet`, `data/raw/contoso/Dim`, `data/raw/contexte`
**Écrit dans** : `data/interim` et `data/interim/dimensions_realignees`

**Pourquoi « flocon » ?** Dans un modèle en étoile, chaque dimension est reliée directement aux ventes. Ici, la géographie n'est reliée aux ventes qu'à travers la boutique, et la catégorie qu'à travers le produit et la sous-catégorie : ce sont des doubles et triples jointures.

**Période** : les trois années étudiées, 2019 à 2021 (2019 est l'année de référence, 2020 et 2021 le périmètre de l'énoncé).

À ce stade, les données sont **transformées mais pas nettoyées**. Le diagnostic (`03_diagnostic`) et le nettoyage (`04_nettoyage`) viennent ensuite.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
# On remonte de dossier en dossier jusqu'à trouver celui qui contient src/ et data/.
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent

# Ajouter la racine aux dossiers où Python cherche les modules
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins, décalage de 12 ans, colonnes de dates à réaligner, codes des pays
#   etl    -> lecture des dimensions, enregistrement dans data/interim
from src import config, etl

config.creer_dossiers()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
print("Racine du projet :", config.ROOT)


# ---------------------------------------------------------------------------
# 3. Petite fonction utilisée pour chaque dimension : garder et renommer des colonnes
# ---------------------------------------------------------------------------
def garder(df, correspondance, table):
    """Garde les colonnes listées dans `correspondance` et les renomme.

    correspondance : dictionnaire  nom Contoso -> nouveau nom (minuscules, sans accent).
    Si une colonne attendue n'existe pas dans la table, on le signale et on continue :
    le notebook ne s'arrête pas pour une colonne secondaire absente.
    """
    presentes = [c for c in correspondance if c in df.columns]
    absentes = [c for c in correspondance if c not in df.columns]
    if absentes:
        print(f"{table} : colonnes attendues mais absentes :", absentes)
    return df[presentes].rename(columns=correspondance)

## 1. Chargement

In [ ]:
# ---------------------------------------------------------------------------
# 1. Ventes consolidées, produites par 01_etl
# ---------------------------------------------------------------------------
chemin_ventes = config.INTERIM / "ventes_consolidees.parquet"
if not chemin_ventes.exists():
    # Message clair plutôt qu'une erreur obscure de fichier introuvable
    raise FileNotFoundError(f"{chemin_ventes} introuvable : exécuter d'abord 01_etl")

ventes = pd.read_parquet(chemin_ventes)
print(f"Ventes consolidées : {len(ventes):,} lignes, {ventes.shape[1]} colonnes".replace(",", " "))


# ---------------------------------------------------------------------------
# 2. Tables de dimension brutes (data/raw/contoso/Dim)
# ---------------------------------------------------------------------------
# etl.lire_dimension renvoie (DataFrame, encodage) : on ne garde ici que le DataFrame ([0]).
# DimCurrency n'est pas chargée : une seule devise dans nos ventes (voir 01_etl).
# DimDate non plus : ses dates couvrent 2005-2011, on construit notre propre calendrier (section 6).
tables = ["DimStore", "DimGeography", "DimProduct", "DimProductSubcategory",
          "DimProductCategory", "DimPromotion", "DimChannel"]
dims = {t: etl.lire_dimension(t)[0] for t in tables}

for t, d in dims.items():
    print(f"{t:25s} {len(d):>6} lignes  {d.shape[1]:>3} colonnes")

## 2. Réalignement des dates des dimensions

Nos ventes sont datées de 2019 à 2021, alors que les dimensions Contoso sont datées de 2003 à 2010. L'inventaire a montré que nos dates sont celles de Contoso **décalées de 12 ans**. On applique donc ce décalage à toutes les dates métier des dimensions.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Décaler les colonnes de dates de chaque dimension
# ---------------------------------------------------------------------------
# config.DATES_A_REALIGNER liste, pour chaque table, ses colonnes de dates métier :
#   DimStore     -> OpenDate, CloseDate, LastRemodelDate
#   DimProduct   -> AvailableForSaleDate, StopSaleDate
#   DimPromotion -> StartDate, EndDate
decalage = pd.DateOffset(years=config.DECALAGE_ANNEES)

realignement = []   # tableau de contrôle : période avant et après, pour chaque colonne
for table, colonnes in config.DATES_A_REALIGNER.items():
    for col in colonnes:
        if col not in dims[table].columns:
            print(f"{table}.{col} : colonne absente, ignorée")
            continue

        # Texte -> date. errors="coerce" : une date illisible devient NaT (valeur manquante).
        avant = pd.to_datetime(dims[table][col], errors="coerce")

        # DateOffset(years=12) ajoute 12 années civiles : le 29 février 2008
        # devient bien le 29 février 2020 (2020 est aussi bissextile).
        apres = avant + decalage
        dims[table][col] = apres

        realignement.append({
            "table": table,
            "colonne": col,
            "dates renseignées": int(apres.notna().sum()),
            "avant : min": avant.min().date() if avant.notna().any() else None,
            "avant : max": avant.max().date() if avant.notna().any() else None,
            "après : min": apres.min().date() if apres.notna().any() else None,
            "après : max": apres.max().date() if apres.notna().any() else None,
        })

realignement = pd.DataFrame(realignement)
display(realignement)


# ---------------------------------------------------------------------------
# 2. Vérification : les ventes en promotion tombent-elles dans la période de leur promotion ?
# ---------------------------------------------------------------------------
# Même contrôle que dans l'inventaire, mais cette fois sur les dates RÉALIGNÉES.
# On écarte la promotion 1 (aucune remise), qui couvre toute la période.
promo_dates = dims["DimPromotion"][["PromotionKey", "StartDate", "EndDate"]]
test = ventes[ventes.id_promotion != 1].merge(
    promo_dates, left_on="id_promotion", right_on="PromotionKey", how="inner")

# On ajoute un jour à la date de fin pour inclure toute la dernière journée ;
# inclusive="left" : début inclus, fin exclue.
dans_periode = test.date.between(test.StartDate, test.EndDate + pd.Timedelta(days=1), inclusive="left")
print(f"Ventes en promotion situées dans la période de leur promotion : {100 * dans_periode.mean():.2f} %")

## 3. Dimension boutique : boutique + géographie

La géographie n'a pas de clé dans nos ventes : on l'atteint par la boutique (`DimStore.GeographyKey`). C'est cette jointure qui donne **le pays de chaque boutique**, indispensable pour utiliser les jours fériés et les mesures COVID-19 du bon pays.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Nettoyer les textes de la géographie AVANT la jointure
# ---------------------------------------------------------------------------
geo = dims["DimGeography"].copy()

# Certains noms contiennent des espaces en trop : "Germany" existe aussi sous la forme
# "Germany " (espace finale). Pour l'ordinateur, ce sont deux pays différents.
# str.strip() retire les espaces en début et en fin de texte.
pays_avant = geo.RegionCountryName.nunique()
for col in ["ContinentName", "RegionCountryName", "StateProvinceName", "CityName"]:
    if col in geo.columns:
        geo[col] = geo[col].str.strip()
print(f"Pays distincts dans DimGeography : {pays_avant} avant nettoyage des espaces, {geo.RegionCountryName.nunique()} après")


# ---------------------------------------------------------------------------
# 2. Jointure boutique -> géographie
# ---------------------------------------------------------------------------
# how="left"           : on garde toutes les boutiques, même sans géographie
# validate="many_to_one" : pandas vérifie qu'une GeographyKey n'apparaît qu'une fois
#                          dans la géographie (sinon des boutiques seraient dupliquées)
boutiques = dims["DimStore"].merge(geo, on="GeographyKey", how="left",
                                   validate="many_to_one", suffixes=("", "_geo"))


# ---------------------------------------------------------------------------
# 3. Garder les colonnes utiles et les renommer
# ---------------------------------------------------------------------------
dim_boutique = garder(boutiques, {
    "StoreKey": "id_boutique",
    "StoreName": "nom_boutique",
    "StoreType": "type_boutique",          # magasin, en ligne, catalogue, revendeur
    "Status": "statut",
    "OpenDate": "date_ouverture",          # réalignée de +12 ans
    "CloseDate": "date_fermeture",         # vide si la boutique est toujours ouverte
    "LastRemodelDate": "date_renovation",
    "EmployeeCount": "effectif",
    "SellingAreaSize": "surface",
    "ContinentName": "continent",
    "RegionCountryName": "pays",
    "StateProvinceName": "region",
    "CityName": "ville",
}, "DimStore")

# Code ISO à 2 lettres du pays (config.PAYS_ISO : nom -> (ISO2, ISO3)).
# map() remplace chaque nom de pays par son code ; les pays absents de config restent vides.
dim_boutique["code_pays"] = dim_boutique.pays.map(lambda p: config.PAYS_ISO.get(p, (None, None))[0])

# Indiquer si la boutique apparaît dans nos ventes (56 boutiques sur 306)
dim_boutique["utilisee"] = dim_boutique.id_boutique.isin(ventes.id_boutique)


# ---------------------------------------------------------------------------
# 4. Contrôles
# ---------------------------------------------------------------------------
assert dim_boutique.id_boutique.is_unique, "Des boutiques sont en double après la jointure"
print(f"Boutiques : {len(dim_boutique)} dont {int(dim_boutique.utilisee.sum())} utilisées par nos ventes")

# Pays des boutiques utilisées, avec le nombre de boutiques par pays
pays_utilises = dim_boutique[dim_boutique.utilisee].groupby(["continent", "pays", "code_pays"], dropna=False).size()
display(pays_utilises.rename("boutiques").reset_index())

# Un pays utilisé mais sans code ISO n'aura ni jours fériés ni données COVID-19 :
# il faut alors l'ajouter à config.PAYS_ISO.
sans_code = dim_boutique[dim_boutique.utilisee & dim_boutique.code_pays.isna()].pays.unique()
print("Pays utilisés sans code ISO dans config :", list(sans_code) or "aucun")

## 4. Dimension produit : produit + sous-catégorie + catégorie

In [ ]:
# ---------------------------------------------------------------------------
# 1. Aplatir la hiérarchie en une seule table
# ---------------------------------------------------------------------------
# produit -> sous-catégorie (double jointure) -> catégorie (triple jointure).
# suffixes : si deux tables ont une colonne du même nom (ETLLoadID, LoadDate...),
# celle de droite reçoit le suffixe indiqué, pour ne pas les confondre.
produits = (
    dims["DimProduct"]
    .merge(dims["DimProductSubcategory"], on="ProductSubcategoryKey", how="left",
           validate="many_to_one", suffixes=("", "_sc"))
    .merge(dims["DimProductCategory"], on="ProductCategoryKey", how="left",
           validate="many_to_one", suffixes=("", "_cat"))
)

# ---------------------------------------------------------------------------
# 2. Garder les colonnes utiles et les renommer
# ---------------------------------------------------------------------------
dim_produit = garder(produits, {
    "ProductKey": "id_produit",
    "ProductName": "nom_produit",
    "ProductCategoryName": "categorie",
    "ProductSubcategoryName": "sous_categorie",
    "BrandName": "marque",
    "Manufacturer": "fabricant",
    "ClassName": "classe",                       # gamme : économique, standard, luxe
    "ColorName": "couleur",
    "UnitCost": "cout_catalogue",                # coût de référence du produit
    "UnitPrice": "prix_catalogue",               # prix de référence, à comparer aux ventes (diagnostic)
    "AvailableForSaleDate": "date_mise_en_vente",   # réalignée de +12 ans
    "StopSaleDate": "date_fin_vente",
    "Status": "statut",
}, "DimProduct")

dim_produit["utilise"] = dim_produit.id_produit.isin(ventes.id_produit)

# ---------------------------------------------------------------------------
# 3. Contrôles
# ---------------------------------------------------------------------------
assert dim_produit.id_produit.is_unique, "Des produits sont en double après les jointures"
print(f"Produits : {len(dim_produit)} dont {int(dim_produit.utilise.sum())} utilisés par nos ventes")

# Nombre de produits utilisés par catégorie
display(dim_produit[dim_produit.utilise].groupby("categorie").size().rename("produits utilisés").reset_index())

## 5. Dimensions promotion et canal

In [ ]:
# ---------------------------------------------------------------------------
# 1. Promotion
# ---------------------------------------------------------------------------
dim_promotion = garder(dims["DimPromotion"], {
    "PromotionKey": "id_promotion",
    "PromotionName": "nom_promotion",
    "PromotionType": "type_promotion",              # ex. remise saisonnière
    "PromotionCategory": "categorie_promotion",     # ex. boutique, revendeur
    "DiscountPercent": "taux_remise_annonce",       # 0.2 = 20 %
    "StartDate": "date_debut",                      # réalignée de +12 ans
    "EndDate": "date_fin",
}, "DimPromotion")

dim_promotion["utilisee"] = dim_promotion.id_promotion.isin(ventes.id_promotion)
assert dim_promotion.id_promotion.is_unique, "Des promotions sont en double"

# Les promotions utilisées par nos ventes, avec leur période réalignée
display(dim_promotion[dim_promotion.utilisee].drop(columns="utilisee"))


# ---------------------------------------------------------------------------
# 2. Canal de vente
# ---------------------------------------------------------------------------
dim_canal = garder(dims["DimChannel"], {
    "ChannelKey": "id_canal",
    "ChannelName": "canal",
}, "DimChannel")
assert dim_canal.id_canal.is_unique, "Des canaux sont en double"
display(dim_canal)

## 6. Dimension date : notre calendrier 2019-2021

Ce calendrier contient ce qui est **vrai dans tous les pays** : jour de la semaine, semaine, mois, trimestre. Ce qui dépend du pays (jours fériés, mesures COVID-19) est dans la table de contexte, à la section suivante.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Une ligne par jour, du 1er janvier 2019 au 31 décembre 2021
# ---------------------------------------------------------------------------
# On couvre les trois années entières, que tous les jours aient des ventes ou non :
# le calendrier doit rester continu pour les graphiques et la prévision.
dim_date = pd.DataFrame({"date": pd.date_range(config.DATE_DEBUT, config.DATE_FIN, freq="D")})

# Noms en français, écrits ici pour ne pas dépendre de la langue de l'ordinateur
NOMS_JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
NOMS_MOIS = ["janvier", "février", "mars", "avril", "mai", "juin",
             "juillet", "août", "septembre", "octobre", "novembre", "décembre"]

# ---------------------------------------------------------------------------
# 2. Colonnes déduites de la date
# ---------------------------------------------------------------------------
d = dim_date.date.dt                              # raccourci vers les propriétés de date
dim_date["annee"] = d.year
dim_date["trimestre"] = d.quarter
dim_date["mois"] = d.month
dim_date["nom_mois"] = d.month.map(lambda m: NOMS_MOIS[m - 1])
dim_date["annee_mois"] = d.strftime("%Y-%m")      # ex. "2020-03", pratique pour trier et regrouper
dim_date["semaine"] = d.isocalendar().week.astype(int).values   # numéro de semaine ISO (1 à 53)
dim_date["jour"] = d.day
dim_date["jour_semaine"] = d.dayofweek + 1        # 1 = lundi ... 7 = dimanche
dim_date["nom_jour"] = d.dayofweek.map(lambda j: NOMS_JOURS[j])
dim_date["fin_de_semaine"] = d.dayofweek >= 5     # samedi (5) ou dimanche (6)

# Indiquer les jours pour lesquels nous avons des ventes (tous, avec les ventes complètes ;
# avec les fichiers du professeur, les jours des 4es trimestres n'en ont pas)
dim_date["avec_ventes"] = dim_date.date.isin(ventes.date)
# Distinguer l'année de référence (2019) des deux années de l'énoncé
dim_date["annee_de_reference"] = dim_date.annee == config.ANNEE_REFERENCE

print(f"Calendrier : {len(dim_date)} jours, du {dim_date.date.min().date()} au {dim_date.date.max().date()}")
print(f"Jours avec des ventes : {int(dim_date.avec_ventes.sum())}")
dim_date.head()

## 7. Contexte par pays et par jour

Les jours fériés et les mesures COVID-19 dépendent du pays. Plutôt qu'un calendrier par pays, on construit **une seule table** avec une colonne `pays` : une ligne par pays **et** par jour.

| pays | date | est_ferie | nom_ferie | severite_covid |
|---|---|---|---|---|
| France | 2020-05-01 | True | Fête du Travail | 87,9 |
| Germany | 2020-05-01 | True | Erster Mai | 76,8 |

Les fichiers sources sont produits par `src/contexte.py` dans `data/raw/contexte`. S'ils sont absents, le notebook continue et laisse les colonnes vides.

**Le cas de 2019.** L'indice de sévérité d'Oxford n'existe qu'à partir du 1er janvier 2020 : il n'y a aucune ligne pour 2019 dans le fichier source. Ce n'est pas une donnée manquante au sens habituel, puisqu'aucune mesure sanitaire n'existait avant 2020. On inscrit donc **0** (aucune restriction) pour tous les jours antérieurs au début de l'indice. Rappel : nos ventes sont en réalité celles de 2007 à 2009, cet indice ne décrit donc pas ce que les boutiques ont vécu. Il sert à vérifier que les ventes n'y réagissent pas (`03_diagnostic`).

In [ ]:
# ---------------------------------------------------------------------------
# 1. La grille complète : chaque pays utilisé x chaque jour du calendrier
# ---------------------------------------------------------------------------
pays_liste = sorted(dim_boutique[dim_boutique.utilisee].pays.dropna().unique())

# MultiIndex.from_product construit toutes les combinaisons (pays, date) ;
# to_frame(index=False) en fait un tableau ordinaire à deux colonnes.
contexte = pd.MultiIndex.from_product(
    [pays_liste, dim_date.date], names=["pays", "date"]).to_frame(index=False)
print(f"Grille : {len(pays_liste)} pays x {len(dim_date)} jours = {len(contexte):,} lignes".replace(",", " "))


# ---------------------------------------------------------------------------
# 2. Jours fériés (data/raw/contexte/jours_feries.csv)
# ---------------------------------------------------------------------------
chemin_feries = config.CONTEXTE / config.FICHIER_JOURS_FERIES
if chemin_feries.exists():
    feries = pd.read_csv(chemin_feries, parse_dates=["date"])
    # Sécurité : un seul nom par (pays, date). Si deux lignes existent pour le même jour,
    # on réunit leurs noms, sinon la jointure dupliquerait des lignes de la grille.
    feries = feries.groupby(["pays", "date"], as_index=False).nom_ferie.agg("; ".join)
    contexte = contexte.merge(feries, on=["pays", "date"], how="left", validate="one_to_one")
    # Un jour est férié si un nom de fête lui a été rattaché
    contexte["est_ferie"] = contexte.nom_ferie.notna()
else:
    print(f"{chemin_feries.name} absent : lancer python -m src.contexte. Colonnes laissées vides.")
    contexte["nom_ferie"] = pd.NA
    contexte["est_ferie"] = pd.NA


# ---------------------------------------------------------------------------
# 3. Sévérité des mesures COVID-19 (data/raw/contexte/severite_covid.csv)
# ---------------------------------------------------------------------------
# Indice d'Oxford (OxCGRT), de 0 (aucune restriction) à 100 (restrictions maximales).
chemin_covid = config.CONTEXTE / config.FICHIER_SEVERITE_COVID
if chemin_covid.exists():
    covid = pd.read_csv(chemin_covid, parse_dates=["date"])
    covid = covid.drop_duplicates(["pays", "date"])
    contexte = contexte.merge(covid, on=["pays", "date"], how="left", validate="one_to_one")
    # Avant le début de l'indice (1er janvier 2020), aucune mesure n'existait : la sévérité vaut 0.
    # On ne touche qu'aux jours antérieurs au premier jour de l'indice : un vide après cette date resterait un vide.
    debut_indice = covid.date.min()
    avant_indice = contexte.date < debut_indice
    print(f"Indice COVID-19 disponible à partir du {debut_indice.date()} : {int(avant_indice.sum()):,} lignes antérieures mises à 0".replace(",", " "))
    contexte.loc[avant_indice, "severite_covid"] = 0.0
else:
    print(f"{chemin_covid.name} absent : télécharger le fichier OxCGRT puis lancer python -m src.contexte. Colonne laissée vide.")
    contexte["severite_covid"] = np.nan


# ---------------------------------------------------------------------------
# 4. Contrôle par pays
# ---------------------------------------------------------------------------
controle_contexte = contexte.groupby("pays").agg(
    jours=("date", "size"),
    jours_feries=("est_ferie", lambda s: int(s.fillna(False).sum())),
    jours_avec_indice_covid=("severite_covid", lambda s: int(s.notna().sum())),
    severite_moyenne=("severite_covid", "mean"),
).round(1)
display(controle_contexte)

## 8. Jointure aux ventes

Chaque vente reçoit les attributs de sa boutique, de son produit, de sa promotion et de son canal, puis le contexte de **son pays ce jour-là**. Aucune vente ne doit être perdue ni dupliquée.

In [ ]:
lignes_avant = len(ventes)

# ---------------------------------------------------------------------------
# 1. Colonnes de chaque dimension à rattacher aux ventes
# ---------------------------------------------------------------------------
# On ne rattache que les attributs utiles à l'analyse. Les tables de dimension complètes
# restent disponibles dans data/interim/dimensions_realignees pour Power BI.
cols_boutique = ["id_boutique", "nom_boutique", "type_boutique", "surface", "effectif",
                 "continent", "pays", "code_pays", "date_ouverture", "date_fermeture"]
cols_produit = ["id_produit", "nom_produit", "categorie", "sous_categorie", "marque", "classe",
                "prix_catalogue", "cout_catalogue"]
cols_promotion = ["id_promotion", "nom_promotion", "type_promotion", "categorie_promotion",
                  "taux_remise_annonce", "date_debut", "date_fin"]

# Ne garder que les colonnes réellement présentes (au cas où une colonne aurait été absente)
def existantes(df, colonnes):
    return [c for c in colonnes if c in df.columns]

# ---------------------------------------------------------------------------
# 2. Jointures successives, toutes "à gauche" : on garde toutes les ventes
# ---------------------------------------------------------------------------
# validate="many_to_one" : plusieurs ventes pour une boutique, mais une seule ligne
# par boutique dans la dimension. Si ce n'était pas le cas, pandas lèverait une erreur.
jointes = (
    ventes
    .merge(dim_boutique[existantes(dim_boutique, cols_boutique)], on="id_boutique", how="left", validate="many_to_one")
    .merge(dim_produit[existantes(dim_produit, cols_produit)], on="id_produit", how="left", validate="many_to_one")
    .merge(dim_promotion[existantes(dim_promotion, cols_promotion)], on="id_promotion", how="left", validate="many_to_one")
    .merge(dim_canal, on="id_canal", how="left", validate="many_to_one")
)

# Renommer les dates de promotion pour ne pas les confondre avec la date de la vente
jointes = jointes.rename(columns={"date_debut": "promo_date_debut", "date_fin": "promo_date_fin"})

# ---------------------------------------------------------------------------
# 3. Contexte : jointure sur DEUX clés, le pays de la boutique et la date de la vente
# ---------------------------------------------------------------------------
# Les deux colonnes de dates doivent avoir exactement le même type pour se correspondre
jointes["date"] = jointes.date.astype("datetime64[ns]")
contexte["date"] = contexte.date.astype("datetime64[ns]")
jointes = jointes.merge(contexte, on=["pays", "date"], how="left", validate="many_to_one")


# ---------------------------------------------------------------------------
# 4. Contrôles : aucune vente perdue ni dupliquée, et toutes rattachées
# ---------------------------------------------------------------------------
assert len(jointes) == lignes_avant, "Les jointures ont changé le nombre de ventes"
assert jointes.id_vente.is_unique, "Des ventes ont été dupliquées par les jointures"

# Pour chaque jointure, une colonne témoin : si elle est vide, la vente n'a pas trouvé
# de correspondance dans la dimension.
temoins = {"boutique": "nom_boutique", "pays": "pays", "produit": "nom_produit",
           "catégorie": "categorie", "promotion": "nom_promotion", "canal": "canal",
           "jours fériés": "est_ferie", "sévérité COVID-19": "severite_covid"}
rattachement = pd.DataFrame([{
    "information": nom,
    "ventes sans correspondance": int(jointes[col].isna().sum()),
    "% rattaché": round(100 * jointes[col].notna().mean(), 2),
} for nom, col in temoins.items() if col in jointes.columns])
display(rattachement)

print(f"Table jointe : {len(jointes):,} lignes, {jointes.shape[1]} colonnes".replace(",", " "))

## 9. Enregistrement dans data/interim

In [ ]:
# ---------------------------------------------------------------------------
# 1. Dimensions réalignées -> data/interim/dimensions_realignees/
# ---------------------------------------------------------------------------
# Une table par dimension, complète (y compris les membres non utilisés par nos ventes) :
# c'est ce que Power BI utilisera pour construire le modèle en flocon.
fichiers_ecrits = []
for nom, table in [("dim_boutique", dim_boutique), ("dim_produit", dim_produit),
                   ("dim_promotion", dim_promotion), ("dim_canal", dim_canal)]:
    chemin = etl.enregistrer_interim(table, nom, sous_dossier="dimensions_realignees")
    fichiers_ecrits.append({"fichier": str(chemin.relative_to(config.DATA)), "lignes": len(table), "colonnes": table.shape[1]})

# ---------------------------------------------------------------------------
# 2. Calendrier, contexte et ventes jointes -> data/interim/
# ---------------------------------------------------------------------------
for nom, table in [("dim_date", dim_date), ("contexte_pays_date", contexte), ("ventes_jointes", jointes)]:
    chemin = etl.enregistrer_interim(table, nom)
    fichiers_ecrits.append({"fichier": str(chemin.relative_to(config.DATA)), "lignes": len(table), "colonnes": table.shape[1]})

display(pd.DataFrame(fichiers_ecrits))

# ---------------------------------------------------------------------------
# 3. Relire les ventes jointes pour vérifier le fichier
# ---------------------------------------------------------------------------
relu = pd.read_parquet(config.INTERIM / "ventes_jointes.parquet")
assert len(relu) == len(jointes), "Le fichier relu n'a pas le même nombre de lignes"
print(f"ventes_jointes relu : {len(relu):,} lignes".replace(",", " "))
print(f"Chiffre d'affaires total : {relu.montant_ventes.sum():,.0f} $".replace(",", " "))

## 10. Le modèle complet : cinq tables de faits

Ce notebook construit le flocon des **ventes**. Mais les ventes ne sont plus la seule table de faits du projet. Quatre autres tables décrivent les mêmes boutiques et les mêmes produits, à un niveau de détail différent. Elles ne peuvent pas devenir des colonnes de la table des ventes : un loyer mensuel ou un stock hebdomadaire n'appartient à aucune vente en particulier.

| Table de faits | Une ligne représente | Rythme | Préparée dans |
|---|---|---|---|
| **Ventes** | un produit, une boutique, un jour, une promotion | jour | `01_etl` à `05_variables` |
| **Charges** (plan stratégique) | une boutique, un mois, un compte, un scénario, une catégorie | mois | `05b_charges` |
| **Objectifs de vente** | une boutique, un produit, un mois, un scénario | mois | `05c_stock_objectifs` |
| **Stock** | une boutique, un produit, une semaine | semaine | `05c_stock_objectifs` |
| **Ventes en ligne détaillées** | une ligne de commande d'un client de la boutique en ligne | jour | `05c_stock_objectifs` |

Ce qui les relie, ce sont les **dimensions partagées** :

| Dimension | Ventes | Charges | Objectifs | Stock | Ventes en ligne |
|---|---|---|---|---|---|
| Boutique (et sa géographie) | oui | oui | oui | oui | oui (une seule) |
| Produit (sous-catégorie, catégorie) | oui | par catégorie | oui | oui | oui |
| Date | jour | mois | mois | semaine | jour |
| Promotion | oui | | | | oui |
| Canal | oui | | oui | | |
| Scénario (réel, budget, prévision) | | oui | oui | | |
| Compte | | oui | | | |
| Client | | | | | oui |

**Conséquence pratique** : deux tables de faits ne se joignent jamais ligne à ligne. On les ramène d'abord au même niveau (boutique et mois, ou boutique, produit et mois), puis on les compare. Comme les rythmes diffèrent (jour, semaine, mois), le niveau commun le plus fin est le **mois**.

Les dimensions réalignées écrites ici (`dim_boutique`, `dim_produit`, `dim_promotion`, `dim_canal`, `dim_date`) servent aux cinq tables.

## Suite

`03_diagnostic` lira `data/interim/ventes_jointes.parquet` pour chercher les anomalies : montants incohérents, retours supérieurs aux ventes, ventes hors de la période de leur promotion, ventes d'une boutique fermée, prix différents du catalogue.